# BERT Fine-Tuning

## Objective

The objective of this notebook is to fine-tune a pre-trained BERT model for four-class mental health text classification.

In this notebook, we will:

- load the processed datasets,
- recreate the tokenizer, Dataset objects, and DataLoaders,
- load a pre-trained BERT classification model,
- configure the device and optimizer,
- train the model,
- evaluate performance on the validation set,
- and save the best-performing model.

## 1. Import Libraries

This section imports the libraries required for data loading, reproducibility, model training, optimization, and evaluation.

In [1]:
from pathlib import Path

import random
import numpy as np
import pandas as pd
import torch

from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

## 2. Reproducibility

Random seeds are set for Python, NumPy, and PyTorch to make the training process as reproducible as possible.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Project Configuration

Reusable project paths and model hyperparameters are defined in one place.

In [3]:
PROJECT_ROOT = Path("..")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_CACHE_DIR = PROJECT_ROOT / "models" / "huggingface_cache"
OUTPUT_MODEL_DIR = PROJECT_ROOT / "models" / "bert_mental_health"

MODEL_NAME = "bert-base-uncased"

NUM_LABELS = 4
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

OUTPUT_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

## 4. Device Selection

The fastest available device is selected for model training.

The notebook supports:

- CUDA for NVIDIA GPUs,
- MPS for Apple Silicon,
- and CPU as a fallback.

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Selected device:", device)

Selected device: mps


## 5. Load the Processed Datasets

The processed training, validation, and test datasets created in Notebook 02 are loaded for fine-tuning.

In [5]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")

validation_df = pd.read_csv(PROCESSED_DATA_DIR / "validation.csv")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")

print("Training set:", train_df.shape)
print("Validation set:", validation_df.shape)
print("Test set:", test_df.shape)

Training set: (34254, 2)
Validation set: (7340, 2)
Test set: (7341, 2)


## 6. Create Label Mappings

Consistent mappings are created between the class names and numerical label IDs.

In [6]:
label2id = {
    "Anxiety": 0,
    "Depression": 1,
    "Normal": 2,
    "Suicidal": 3,
}

id2label = {label_id: label_name for label_name, label_id in label2id.items()}

for dataframe in [
    train_df,
    validation_df,
    test_df,
]:
    dataframe["label"] = dataframe["status"].map(label2id)

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'Anxiety': 0, 'Depression': 1, 'Normal': 2, 'Suicidal': 3}
id2label: {0: 'Anxiety', 1: 'Depression', 2: 'Normal', 3: 'Suicidal'}


## 7. Load the BERT Tokenizer

The tokenizer associated with `bert-base-uncased` is loaded from the Hugging Face model hub or the local cache.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_DIR,
)

print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded: BertTokenizer
Vocabulary size: 30522


## 8. Create the PyTorch Dataset Class

A custom PyTorch Dataset converts each text and label into the tensors required by BERT.

In [8]:
class MentalHealthDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length: int = 128,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        text = self.dataframe.iloc[index]["text"]
        label = self.dataframe.iloc[index]["label"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": (encoding["input_ids"].squeeze(0)),
            "attention_mask": (encoding["attention_mask"].squeeze(0)),
            "labels": torch.tensor(
                label,
                dtype=torch.long,
            ),
        }

## 9. Create Dataset and DataLoader Objects

In [9]:
train_dataset = MentalHealthDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

validation_dataset = MentalHealthDataset(
    dataframe=validation_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_dataset = MentalHealthDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [10]:
batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print(
    "Attention mask:",
    batch["attention_mask"].shape,
)
print("Labels:", batch["labels"].shape)

Input IDs: torch.Size([16, 128])
Attention mask: torch.Size([16, 128])
Labels: torch.Size([16])


## 10. Load the BERT Classification Model

A pre-trained `bert-base-uncased` model is loaded with a new classification head for the four target classes.

The BERT encoder starts with pre-trained language knowledge, while the classification head is adapted to this project.

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    cache_dir=MODEL_CACHE_DIR,
)

model = model.to(device)

print("Model loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("Classifier layer:")
print(model.classifier)
print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.
Number of labels: 4
Classifier layer:
Linear(in_features=768, out_features=4, bias=True)
Model device: mps:0


## 11. Configure the Optimizer

AdamW is used to update the trainable model parameters during fine-tuning.

A small learning rate is selected because the model is already pre-trained and its existing language representations should be adjusted carefully.

In [12]:
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])

Optimizer: AdamW
Learning rate: 2e-05


## 12. Test a Forward Pass

One batch is passed through the model before training to confirm that the model, tensors, labels, and selected device are compatible.

In [13]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

model.train()

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 1.4025206565856934
Logits shape: torch.Size([16, 4])


## 13. Define Training and Validation Functions

Separate functions are created for training and validation.

The training function performs forward propagation, loss calculation, backpropagation, and parameter updates.

The validation function evaluates the model without calculating gradients or updating parameters.

In [14]:
def train_one_epoch(
    model,
    data_loader,
    optimizer,
    device,
    max_batches=None,
):
    model.train()

    total_loss = 0.0
    processed_batches = 0

    for batch_index, batch in enumerate(data_loader):
        if max_batches is not None and batch_index >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        processed_batches += 1

    return total_loss / processed_batches

## 14. Fine-Tune the Model

The model is trained for multiple epochs.

After each epoch, validation loss and validation accuracy are calculated. The model with the lowest validation loss is saved.

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    cache_dir=MODEL_CACHE_DIR,
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Model and optimizer reset.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and optimizer reset.


In [16]:
def evaluate_model(
    model,
    data_loader,
    device,
    max_batches=None,
):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    processed_batches = 0

    with torch.no_grad():
        for batch_index, batch in enumerate(data_loader):
            if max_batches is not None and batch_index >= max_batches:
                break

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            predictions = torch.argmax(
                outputs.logits,
                dim=1,
            )

            total_loss += outputs.loss.item()

            correct_predictions += (predictions == labels).sum().item()

            total_predictions += labels.size(0)
            processed_batches += 1

    average_loss = total_loss / processed_batches
    accuracy = correct_predictions / total_predictions

    return average_loss, accuracy

In [17]:
smoke_train_loss = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    max_batches=100,
)

smoke_validation_loss, smoke_validation_accuracy = evaluate_model(
    model=model,
    data_loader=validation_loader,
    device=device,
    max_batches=50,
)

print(f"Smoke training loss: {smoke_train_loss:.4f}")
print(f"Smoke validation loss: {smoke_validation_loss:.4f}")
print(f"Smoke validation accuracy: " f"{smoke_validation_accuracy:.4f}")

Smoke training loss: 1.0592
Smoke validation loss: 0.8419
Smoke validation accuracy: 0.6587


In [18]:
model.save_pretrained(OUTPUT_MODEL_DIR)

tokenizer.save_pretrained(OUTPUT_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/bert_mental_health/tokenizer_config.json',
 '../models/bert_mental_health/tokenizer.json')

## Key Takeaways

In this notebook:

- a pre-trained BERT model was loaded for four-class text classification,
- Apple MPS acceleration was configured,
- the AdamW optimizer was initialized,
- training and validation functions were implemented,
- a forward pass was successfully tested,
- and a limited smoke test verified that loss calculation, backpropagation, parameter updates, and validation work correctly.

The fine-tuning pipeline is now ready for a full training run.

In [19]:
print("Fine-tuning pipeline validated successfully.")
print(f"Smoke training loss: {smoke_train_loss:.4f}")
print(f"Smoke validation loss: {smoke_validation_loss:.4f}")
print(f"Smoke validation accuracy: " f"{smoke_validation_accuracy:.4f}")
print("Model device:", next(model.parameters()).device)

Fine-tuning pipeline validated successfully.
Smoke training loss: 1.0592
Smoke validation loss: 0.8419
Smoke validation accuracy: 0.6587
Model device: mps:0


The current checkpoint was generated after a successful smoke test to validate the fine-tuning pipeline.

## Future Work

The next step is to perform full fine-tuning using all training batches and multiple epochs, followed by a comprehensive evaluation on the held-out test set.